
# Extended Lab: Discrete Response Regression Models
### Probit/Logit → Multinomial Logit → Poisson → Negative Binomial

This lab extends the four short demos from *Building Statistical Models in Python* (Ch. 8) into a
single, progressively harder workbook. Each part keeps the original dataset/idea but asks you to go
further: more diagnostics, more realistic data splits, more model-comparison, and a few "what would
you do differently in the real world" extensions.

**How to use this notebook**
- This is the **worked / extended** version — it contains guidance, some starter code, and a full
  narrative. If you want to test yourself from a blank slate, use `02_Skeleton_Practice.ipynb`.
- Solutions to every exercise flagged with **Exercise** live in `05_Detailed_Solutions.ipynb`.
- A one-page formula/code reference lives in `03_Cheat_Sheet.ipynb`.
- A generic project template you can reuse on your own data lives in `04_Reusable_Template.ipynb`.

**Learning objectives**
1. Explain why OLS is inappropriate for binary/count outcomes and derive the logit/probit link.
2. Fit, interpret, and validate logit and probit models (coefficients, odds ratios, marginal effects).
3. Extend binary classification to multi-class outcomes with MNLogit / multinomial softmax.
4. Fit Poisson regression for count data and check the equidispersion assumption.
5. Detect overdispersion and correct for it with negative binomial regression.
6. Compare models using pseudo-R², AIC/BIC, log-likelihood, and out-of-sample error — not just accuracy.


In [ ]:

# Environment check / installs (uncomment if needed)
# %pip install statsmodels scikit-learn pandas numpy matplotlib seaborn scipy requests --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

pd.set_option('display.max_columns', 50)
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid') if 'seaborn-v0_8-whitegrid' in plt.style.available else None



---
## Part 1 — Binary Outcomes: Logit and Probit

### 1.1 Theory recap

For a binary outcome $y \in \{0, 1\}$, linear regression $y = \beta_0 + \beta_1 x + \epsilon$ is a poor
model because predictions are not bounded to $[0, 1]$. Instead we model the **log-odds**:

$$\log\left(\frac{P}{1-P}\right) = \beta_0 + \beta_1 x_1 + \dots + \beta_n x_n = z$$

and recover a probability with the **logistic CDF** (logit model):

$$P = F(z) = \frac{e^z}{1+e^z}$$

or with the **standard normal CDF** (probit model):

$$P = \Phi(z) = \int_{-\infty}^{z} \phi(u)\,du$$

Both are fit by **maximum likelihood** rather than least squares. Logit coefficients are interpreted
in log-odds units (exponentiate for odds ratios); probit coefficients are in z-score units and are
usually interpreted through **marginal effects** instead.


In [ ]:

# 1.2 Recreate the admissions dataset from the book
train = pd.DataFrame({
    'Admitted': [1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0],
    'GPA': [2.8, 3.3, 3.7, 3.7, 3.7, 3.3, 3.7, 3, 1.7, 3.6, 3.3, 4, 3.2, 3.4, 2.8, 4, 1.5, 2.7, 2.3, 2.3, 2.7, 2.2,
            3.3, 3.3, 4, 2.3, 3.6, 3.4, 4, 3.7, 2.3],
    'Exp': [8, 6, 5, 5, 6, 3, 4, 2, 1, 5, 5, 3, 6, 5, 4, 4, 4, 1, 1, 2, 2, 2, 1, 4, 4, 4, 5, 2, 4, 6, 3]
})

test = pd.DataFrame({
    'Admitted': [1, 0, 1, 0, 1],
    'GPA': [2.9, 2.4, 3.8, 3, 3.3],
    'Exp': [9, 1, 6, 1, 4]
})

train.describe()


In [ ]:

# 1.3 Fit the logit model (as in the book)
logit_model = smf.logit('Admitted ~ GPA + Exp', data=train).fit()
print(logit_model.summary())



> **Exercise 1.1 — Odds ratios.**
> Exponentiate the logit coefficients (`np.exp(logit_model.params)`) and write one sentence per
> variable explaining what a one-unit increase does to the *odds* of admission (not the probability).

> **Exercise 1.2 — Fit a probit model.**
> Fit `smf.probit('Admitted ~ GPA + Exp', data=train)` and compare the sign and significance of the
> coefficients to the logit model. Why are the *magnitudes* so different even though the conclusions
> are similar?

> **Exercise 1.3 — Marginal effects.**
> Use `logit_model.get_margeff(at='mean').summary()` (and the probit equivalent) to compute average
> marginal effects. Compare these — they should be much closer between the two models than the raw
> coefficients were.


In [ ]:

# Scaffold for Exercise 1.1-1.3 (fill in / uncomment)

# odds_ratios = np.exp(logit_model.params)
# print(odds_ratios)

# probit_model = smf.probit('Admitted ~ GPA + Exp', data=train).fit()
# print(probit_model.summary())

# print(logit_model.get_margeff(at='mean').summary())
# print(probit_model.get_margeff(at='mean').summary())



### 1.4 Beyond the book: proper evaluation

The book's test set gives 100% accuracy on 5 rows — that is not a meaningful validation. Let's build
a larger synthetic dataset with the same data-generating process so we can evaluate more honestly.


In [ ]:

# 1.5 Simulate a larger, noisier admissions dataset from a *known* logit DGP
rng = np.random.default_rng(7)
n = 800
gpa = rng.uniform(1.5, 4.0, n)
exp = rng.integers(0, 10, n)
true_b0, true_bgpa, true_bexp = -11.0, 2.6, 0.7
z = true_b0 + true_bgpa * gpa + true_bexp * exp
p = 1 / (1 + np.exp(-z))
admitted = rng.binomial(1, p)

sim = pd.DataFrame({'Admitted': admitted, 'GPA': gpa, 'Exp': exp})
sim.head()


In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, accuracy_score, ConfusionMatrixDisplay,
                              roc_curve, roc_auc_score, precision_score, recall_score, f1_score)

sim_train, sim_test = train_test_split(sim, test_size=0.25, random_state=1, stratify=sim['Admitted'])

sim_logit = smf.logit('Admitted ~ GPA + Exp', data=sim_train).fit()
print(sim_logit.summary())

y_prob = sim_logit.predict(sim_test[['GPA', 'Exp']])
y_pred = (y_prob >= 0.5).astype(int)

print('Accuracy :', accuracy_score(sim_test['Admitted'], y_pred))
print('Precision:', precision_score(sim_test['Admitted'], y_pred))
print('Recall   :', recall_score(sim_test['Admitted'], y_pred))
print('F1       :', f1_score(sim_test['Admitted'], y_pred))
print('AUC      :', roc_auc_score(sim_test['Admitted'], y_prob))

cm = confusion_matrix(sim_test['Admitted'], y_pred)
ConfusionMatrixDisplay(cm).plot()
plt.title('Confusion matrix — simulated admissions (test set)')
plt.show()



> **Exercise 1.4 — ROC curve and threshold choice.**
> Plot the ROC curve with `roc_curve`. The default classification threshold is 0.5 — is that
> appropriate here? Discuss a scenario (e.g., a university with limited seats) where you'd pick a
> different threshold, and recompute precision/recall at that new threshold.

> **Exercise 1.5 — Recovering the truth.**
> Compare `sim_logit.params` to the `true_b0, true_bgpa, true_bexp` used to generate the data. Are
> they close? What would make the estimates converge to the true values as `n` grows?

> **Exercise 1.6 — Pseudo-R².**
> Report `sim_logit.prsquared` (McFadden's pseudo-R²). Explain to someone used to OLS $R^2$ why this
> number will *never* reach 1 even for a perfect model, and why it shouldn't be compared to OLS $R^2$
> values directly.


In [ ]:

# Scaffold for Exercise 1.4-1.6

# fpr, tpr, thresholds = roc_curve(sim_test['Admitted'], y_prob)
# plt.plot(fpr, tpr); plt.plot([0,1],[0,1],'--'); plt.xlabel('FPR'); plt.ylabel('TPR'); plt.show()

# print(sim_logit.params)
# print('true:', true_b0, true_bgpa, true_bexp)

# print('McFadden pseudo-R2:', sim_logit.prsquared)



---
## Part 2 — Multinomial Logit: More Than Two Classes

### 2.1 Theory recap

When the outcome has $K > 2$ unordered categories, we generalize the logit model by picking a
baseline category $k_0$ and modeling $K-1$ log-odds equations simultaneously:

$$\log \frac{P(y=k)}{P(y=k_0)} = \beta_{0k} + \beta_{1k} x_1 + \dots, \quad k \neq k_0$$

Probabilities are recovered with the **softmax** function:

$$P(y=k) = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}$$

`statsmodels.MNLogit` and `sklearn.LogisticRegression` (which handles multinomial classification automatically for a multi-class target) both implement this,
and (as the book shows) should agree closely.


In [ ]:

from sklearn import datasets
from sklearn.linear_model import LogisticRegression
import statsmodels.discrete.discrete_model as sm_discrete

iris = datasets.load_iris()
df = pd.DataFrame(iris.data, columns=['sepal_length', 'sepal_width', 'petal_length', 'petal_width'])
df['target'] = iris.target
df.head()


In [ ]:

X = df.drop('target', axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1, stratify=y)

model_sk = LogisticRegression(solver='lbfgs', max_iter=500)  # multinomial is automatic for multi-class y in modern sklearn
model_sk.fit(X_train, y_train)
pred_sk = model_sk.predict(X_test)
print('sklearn test accuracy:', accuracy_score(y_test, pred_sk))

X_train_c = sm.add_constant(X_train)
X_test_c = sm.add_constant(X_test)
model_stat = sm_discrete.MNLogit(y_train, X_train_c).fit(method='bfgs')
print(model_stat.summary())

pred_stat = np.asarray(model_stat.predict(X_test_c)).argmax(1)
print('statsmodels test accuracy:', accuracy_score(y_test, pred_stat))



> **Exercise 2.1 — Relative risk ratios.**
> `MNLogit` coefficients are log-odds *relative to the baseline category* (category 0 by default).
> Exponentiate `model_stat.params` and interpret one coefficient in plain English: "holding other
> features fixed, a 1cm increase in X multiplies the odds of being class k (vs. the baseline class)
> by ___."

> **Exercise 2.2 — Regularization strength.**
> Refit `LogisticRegression` with `C=0.01` and `C=100`. How does the confusion matrix change? Relate
> this to bias/variance tradeoff.

> **Exercise 2.3 — Class imbalance stress-test.**
> Iris is perfectly balanced (50/50/50). Create an artificially imbalanced version (e.g., drop 80% of
> class 2 rows) and refit. Does accuracy still tell the full story? Report per-class precision/recall
> or a macro-F1 in addition to accuracy.

> **Exercise 2.4 — Add a 4th class.**
> Multinomial logit isn't limited to 3 classes. Using `sklearn.datasets.load_wine()` (3 classes) or
> a simulated 4-class dataset, refit and report a normalized confusion matrix.


In [ ]:

# Scaffold for Exercise 2.1-2.4

# rrr = np.exp(model_stat.params)
# print(rrr)

# for C in [0.01, 100]:
#     m = LogisticRegression(solver='lbfgs', C=C, max_iter=500).fit(X_train, y_train)
#     print(C, accuracy_score(y_test, m.predict(X_test)))

# imbalanced = pd.concat([df[df.target != 2], df[df.target == 2].sample(frac=0.2, random_state=1)])
# ...

# from sklearn.datasets import load_wine
# wine = load_wine()



---
## Part 3 — Poisson Regression for Counts

### 3.1 Theory recap

Count outcomes $y \in \{0, 1, 2, \dots\}$ are modeled with the Poisson distribution:

$$P(Y=k) = \frac{\lambda^k e^{-\lambda}}{k!}$$

A **log-linear** model links the mean count to covariates:

$$\ln(\lambda) = \ln(E[y]) = \beta_0 + \beta_1 x_1 + \dots + \beta_n x_n
\quad\Longleftrightarrow\quad \lambda = e^{\beta_0 + \beta_1 x_1 + \dots}$$

**Key assumption**: the Poisson distribution has mean = variance (**equidispersion**). This is the
assumption we stress-test in Part 4.


In [ ]:

# 3.2 Visualize how the Poisson distribution's shape changes with lambda
from scipy.stats import poisson

means = [12, 5, 2]
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for i, m in enumerate(means):
    r = poisson.rvs(m, size=2000, random_state=42)
    labels, counts = np.unique(r, return_counts=True)
    ax[i].bar(labels, counts)
    ax[i].set_title(f'Poisson(λ={m})  mean={r.mean():.2f} var={r.var():.2f}')
plt.tight_layout()
plt.show()



### 3.3 Bike-sharing case study (as in the book)

This section downloads the UCI Bike Sharing dataset and reproduces (and extends) the weekly
casual-rental Poisson model. **Requires internet access** — if you don't have it, skip to 3.4 which
uses a locally-simulated equivalent.


In [ ]:

import shutil, requests
from datetime import datetime

try:
    r = requests.get('https://archive.ics.uci.edu/ml/machine-learning-databases/00275/Bike-Sharing-Dataset.zip', timeout=10)
    with open('./Bike-Sharing-Dataset.zip', 'wb') as f:
        f.write(r.content)
    shutil.unpack_archive('Bike-Sharing-Dataset.zip', './Bike-Sharing-Dataset')
    bike_df = pd.read_csv('./Bike-Sharing-Dataset/day.csv')
    HAVE_BIKE_DATA = True
except Exception as e:
    print('Could not download dataset (offline?). Will use simulated data instead:', e)
    HAVE_BIKE_DATA = False


In [ ]:

if HAVE_BIKE_DATA:
    bike_df['isoweek'] = bike_df.dteday.apply(lambda x: datetime.strptime(x, '%Y-%m-%d').date().isocalendar()[1])
    bike_df['isoyear'] = bike_df.dteday.apply(lambda x: datetime.strptime(x, '%Y-%m-%d').date().isocalendar()[0])
    bike_df = bike_df[bike_df.isoyear == 2011]

    X = bike_df.groupby('isoweek').mean(numeric_only=True)[['atemp', 'season', 'weathersit', 'hum', 'windspeed', 'holiday']]
    X['holiday'] = X['holiday'].apply(lambda x: 1 if x > 0.1 else 0)
    X = sm.add_constant(X)
    y = bike_df.groupby('isoweek').mean(numeric_only=True)['casual']

    poisson_fit = sm.Poisson(y, X).fit()
    print(poisson_fit.summary())



> **Exercise 3.1 — Coefficient interpretation.**
> Exponentiate `poisson_fit.params['atemp']`. Interpret it as "a one-unit increase in normalized
> temperature multiplies expected weekly casual rentals by ___, holding other variables fixed."

> **Exercise 3.2 — Deviance and goodness of fit.**
> Report `poisson_fit.deviance` and `poisson_fit.df_resid`. A common rule of thumb: if
> `deviance / df_resid` is much greater than 1, you likely have overdispersion. Compute this ratio.

> **Exercise 3.3 — Exposure/offset.**
> The book models weekly *mean* daily counts, sidestepping the "7 days per week" exposure issue.
> Real Poisson regression problems (e.g., insurance claims per policy-year, defects per batch size)
> usually need an **offset** term: `ln(y) = ln(exposure) + b0 + b1x1 + ...`. Look up
> `sm.Poisson(y, X, offset=np.log(exposure))` and explain in your own words when an offset is needed
   versus when it isn't.


In [ ]:

# 3.4 Offline fallback / offset exercise scaffold: simulate insurance-claims-style count data
rng = np.random.default_rng(11)
n = 500
exposure = rng.uniform(0.5, 2.0, n)          # policy-years
driver_age = rng.integers(18, 75, n)
prior_claims = rng.integers(0, 4, n)

log_rate = -3.0 - 0.01 * (driver_age - 40) + 0.35 * prior_claims
expected_claims = exposure * np.exp(log_rate)
claims = rng.poisson(expected_claims)

ins = pd.DataFrame({'claims': claims, 'exposure': exposure, 'driver_age': driver_age, 'prior_claims': prior_claims})
Xi = sm.add_constant(ins[['driver_age', 'prior_claims']])
offset_model = sm.Poisson(ins['claims'], Xi, offset=np.log(ins['exposure'])).fit()
print(offset_model.summary())



---
## Part 4 — Overdispersion and Negative Binomial Regression

### 4.1 Theory recap

Poisson regression assumes $\text{Var}(y) = E[y] = \mu$. Real count data is very often
**overdispersed**: $\text{Var}(y) > \mu$. Fitting Poisson anyway gives *correct point estimates* but
**understated standard errors** — you'll see too many "significant" variables.

Negative binomial regression adds a dispersion parameter $\alpha$:

$$\text{Var}(y) = \mu + \alpha \mu^2$$

When $\alpha \to 0$, this collapses back to Poisson. $\alpha$ is typically estimated first via an
**auxiliary OLS regression** (Cameron & Trivedi's approach, as in the book), then plugged into
`sm.GLM(..., family=NegativeBinomial(alpha=...))`, or estimated jointly via
`sm.NegativeBinomial(...).fit()` / `smf.glm(..., family=sm.families.NegativeBinomial())`.


In [ ]:

# 4.2 Reproduce the book's affairs/children example
data = sm.datasets.fair.load().data
data = sm.add_constant(data, prepend=False)

print('Mean count of children:', data['children'].mean())
print('Variance of children  :', data['children'].var())

plt.hist(data['children'], bins=30)
plt.xlabel('Child count'); plt.title('Distribution of children per marriage')
plt.show()


In [ ]:

y = round(data['children'])
X = data[['const', 'age', 'religious', 'yrs_married', 'educ', 'occupation', 'occupation_husb', 'affairs', 'rate_marriage']]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1)

# Step 1: fit Poisson to get mu_hat
poisson_model = sm.GLM(y_train, X_train, family=sm.families.Poisson()).fit()

# Step 2: auxiliary OLS regression for alpha (Cameron & Trivedi approach)
df_aux = pd.DataFrame()
df_aux['y_mu_hat'] = poisson_model.mu
df_aux['children'] = y_train.values
df_aux['y_auxiliary'] = ((df_aux['children'] - df_aux['y_mu_hat'])**2 - df_aux['y_mu_hat']) / df_aux['y_mu_hat']

ols_model = smf.ols('y_auxiliary ~ y_mu_hat - 1', df_aux).fit()
print(ols_model.summary())
alpha_hat = ols_model.params.iloc[0]
print('\nEstimated alpha:', alpha_hat)


In [ ]:

from statsmodels.genmod.families.family import NegativeBinomial

nb_model = sm.GLM(y_train, X_train, family=NegativeBinomial(alpha=alpha_hat)).fit()
print(nb_model.summary())

from sklearn.metrics import mean_squared_error
def MSE(y_true, y_pred, squared=True):
    val = mean_squared_error(y_true, y_pred)
    return val if squared else np.sqrt(val)
print('Poisson  train RMSE:', MSE(y_train, poisson_model.predict(X_train), squared=False))
print('Poisson  test  RMSE:', MSE(y_test, poisson_model.predict(X_test), squared=False))
print('NegBin   train RMSE:', MSE(y_train, nb_model.predict(X_train), squared=False))
print('NegBin   test  RMSE:', MSE(y_test, nb_model.predict(X_test), squared=False))



> **Exercise 4.1 — Compare standard errors.**
> Put `poisson_model` and `nb_model` standard errors for each coefficient side by side in one
> DataFrame. Which coefficients "lose" significance once overdispersion is properly accounted for?

> **Exercise 4.2 — Statistical test for overdispersion.**
> The auxiliary regression's t-statistic on `y_mu_hat` *is* a formal test of $H_0: \alpha = 0$. State
> the conclusion at $\alpha=0.05$ (pun intended) significance and explain what it means practically.

> **Exercise 4.3 — AIC/BIC comparison.**
> Compare `poisson_model.aic`/`.bic` (Note: `sm.GLM` also exposes these) against a proper
> `sm.NegativeBinomial(y_train, X_train).fit()` fit's `.aic`/`.bic`. Lower is better — which model
   wins, and does that match your conclusion from Exercise 4.2?

> **Exercise 4.4 — Simulate your own overdispersed data.**
> Using `scipy.stats.nbinom`, simulate a count variable with mean 10 and variance 40 (solve for `n`,
> `p` first), fit both a Poisson and a negative binomial model, and show that only the negative
> binomial model recovers the correct variance behavior via its predicted confidence intervals.


In [ ]:

# Scaffold for Exercise 4.1-4.4

# se_compare = pd.DataFrame({'poisson_se': poisson_model.bse, 'negbin_se': nb_model.bse})
# se_compare['ratio'] = se_compare['negbin_se'] / se_compare['poisson_se']
# print(se_compare)

# nb2_model = sm.NegativeBinomial(y_train, X_train).fit()
# print('Poisson AIC/BIC:', poisson_model.aic, poisson_model.bic)
# print('NegBin  AIC/BIC:', nb2_model.aic, nb2_model.bic)

# from scipy.stats import nbinom
# target_mean, target_var = 10, 40
# p_param = target_mean / target_var
# n_param = target_mean**2 / (target_var - target_mean)
# sim_counts = nbinom.rvs(n_param, p_param, size=2000, random_state=1)
# print(sim_counts.mean(), sim_counts.var())



---
## Part 5 — Capstone: Choosing the Right Model

You are given a mystery count variable below. Work through the standard decision process:

1. Plot the distribution. Is it discrete non-negative counts?
2. Compare mean vs. variance — equidispersed or overdispersed?
3. Fit Poisson. Check `deviance/df_resid` and/or an auxiliary OLS test for `alpha`.
4. If overdispersed, refit as negative binomial and compare AIC/BIC and standard errors.
5. Write a 3-5 sentence recommendation memo: which model would you ship to production, and why?


In [ ]:

# Mystery dataset generator — do not peek at the parameters until after you've done the analysis!
def _make_mystery_data(seed=99, n=600):
    rng = np.random.default_rng(seed)
    x1 = rng.normal(0, 1, n)
    x2 = rng.integers(0, 5, n)
    mu = np.exp(0.5 + 0.8 * x1 + 0.3 * x2)
    # Poisson-Gamma mixture => negative-binomial-like overdispersion
    gamma_noise = rng.gamma(shape=2.0, scale=0.5, size=n)
    y = rng.poisson(mu * gamma_noise)
    return pd.DataFrame({'y': y, 'x1': x1, 'x2': x2})

mystery = _make_mystery_data()
mystery.head()



> **Exercise 5.1 — Full workflow.** Carry out steps 1-5 above on `mystery`. Compare your final choice
> and write-up against the solutions notebook.


In [ ]:

# Your workspace for the capstone
# 1. Distribution
# 2. mean vs variance
# 3. Poisson fit + dispersion check
# 4. Negative binomial fit + comparison
# 5. Recommendation



---
## Wrap-up

You've now covered the full arc from the book chapter, plus:
- proper train/test evaluation with simulated ground truth for the logit model,
- ROC/AUC and threshold selection,
- multinomial logit interpretation, regularization, and class-imbalance handling,
- Poisson regression with an **offset/exposure** term (a very common real-world requirement the book
  doesn't cover),
- a formal overdispersion test and AIC/BIC-based model comparison for negative binomial regression,
- an end-to-end "unknown dataset" capstone exercise.

Continue to `05_Detailed_Solutions.ipynb` for full worked solutions, or `04_Reusable_Template.ipynb`
to start a new project with this same structure.
